In [2]:
!pip install -U transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 69.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 93.1 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.3
    Uninstalling datasets-4.8.3:
      Successfully uninstalled datasets-4.8.3
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate

In [3]:
!pip install librosa soundfile audiomentations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 10.9 MB/s eta 0:00:00


In [5]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE = Path("/kaggle/working/ser-india")
RAW  = BASE / "data/raw"
PROC = BASE / "data/processed"

for p in [RAW/"shemo", RAW/"iiith", PROC]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders ready")

Folders ready


In [6]:
SHEMO = RAW / "shemo_repo"
SHEMO.mkdir(parents=True, exist_ok=True)

print("Shemo folder created at:", SHEMO)

Shemo folder created at: /kaggle/working/ser-india/data/raw/shemo_repo


In [7]:
import shutil
from pathlib import Path

src_paths = [
    Path("/kaggle/input/datasets/apoorv007/shemorepo/female"),
    Path("/kaggle/input/datasets/apoorv007/shemorepo/male")
]

# Copy folders
for src in src_paths:
    dst = SHEMO / src.name
    shutil.copytree(src, dst, dirs_exist_ok=True)

print("Files copied to:", SHEMO)

Files copied to: /kaggle/working/ser-india/data/raw/shemo_repo


In [8]:
shemo_repo = RAW / "shemo_repo"
wav_files = list(shemo_repo.rglob("*.wav"))
print(f"Total WAV files found: {len(wav_files)}")
print("\nSample filenames:")
for f in wav_files[:8]:
    print(f"  {f.name}  →  parent: {f.parent.name}")

Total WAV files found: 3000

Sample filenames:
  M02A27.wav  →  parent: male
  M51N10.wav  →  parent: male
  M37N14.wav  →  parent: male
  M34H03.wav  →  parent: male
  M02N10.wav  →  parent: male
  M20N08.wav  →  parent: male
  M18N14.wav  →  parent: male
  M43A10.wav  →  parent: male


In [9]:
import pandas as pd
from pathlib import Path

data = []

for file in wav_files:
    filename = file.name
    
    # Extract emotion code (example: N, H, A, S, F, D)
    emotion_code = filename[3]  # check this based on your dataset
    
    data.append({
        "path": str(file),
        "emotion": emotion_code
    })

df = pd.DataFrame(data)
df.head()

,path,emotion
0,/kaggle/working/ser-india/data/raw/shemo_repo/...,A
1,/kaggle/working/ser-india/data/raw/shemo_repo/...,N
2,/kaggle/working/ser-india/data/raw/shemo_repo/...,N
3,/kaggle/working/ser-india/data/raw/shemo_repo/...,H
4,/kaggle/working/ser-india/data/raw/shemo_repo/...,N


Arrowhead DOES NOT care about:

happiness
disgust
Map to business classes:

In [10]:
mapping = {
    'A': 'frustrated',   # anger
    'N': 'calm',         # neutral
    'H': 'calm',         # happiness
    'W': 'disengaged'    # surprise (best approximation)
}

df["label"] = df["emotion"].map(mapping)
df.to_csv('shemo.csv', index=False)
df.head()

,path,emotion,label
0,/kaggle/working/ser-india/data/raw/shemo_repo/...,A,frustrated
1,/kaggle/working/ser-india/data/raw/shemo_repo/...,N,calm
2,/kaggle/working/ser-india/data/raw/shemo_repo/...,N,calm
3,/kaggle/working/ser-india/data/raw/shemo_repo/...,H,calm
4,/kaggle/working/ser-india/data/raw/shemo_repo/...,N,calm


In [11]:
df = pd.read_csv("/kaggle/working/shemo.csv")   # ← change to your actual CSV path
print(df.head())
print(df.columns.tolist())
print(df['label'].value_counts())       # ← change 'label' to your actual column name if different
print(f"Total samples: {len(df)}")

                                                path emotion       label
0  /kaggle/working/ser-india/data/raw/shemo_repo/...       A  frustrated
1  /kaggle/working/ser-india/data/raw/shemo_repo/...       N        calm
2  /kaggle/working/ser-india/data/raw/shemo_repo/...       N        calm
3  /kaggle/working/ser-india/data/raw/shemo_repo/...       H        calm
4  /kaggle/working/ser-india/data/raw/shemo_repo/...       N        calm
['path', 'emotion', 'label']
label
calm          1229
frustrated    1059
disengaged     225
Name: count, dtype: int64
Total samples: 3000


In [12]:
LABEL2ID = {
    "calm":        0,
    "frustrated":  1,
    "disengaged":  2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print(LABEL2ID)
print(f"Number of classes: {NUM_LABELS}")

{'calm': 0, 'frustrated': 1, 'disengaged': 2}
Number of classes: 3


Phone audio augmentation

In [13]:
def simulate_phone_audio(audio: np.ndarray, sr: int = 16000) -> np.ndarray:
    # Codec compression: downsample to 8kHz then back
    degraded = librosa.resample(audio, orig_sr=sr, target_sr=8000)
    degraded = librosa.resample(degraded, orig_sr=8000, target_sr=sr)

    # Gaussian noise at ~12dB SNR
    signal_power = np.mean(degraded ** 2) + 1e-9
    noise_power  = signal_power / (10 ** (12 / 10))
    noise        = np.random.normal(0, np.sqrt(noise_power), len(degraded))
    degraded     = degraded + noise

    # Clip to safe range
    degraded = np.clip(degraded, -1.0, 1.0)
    return degraded.astype(np.float32)

print("Augmentation function ready")

Augmentation function ready


In [14]:
TARGET_SR      = 16000
MAX_SAMPLES    = TARGET_SR * 6   # 6 seconds max per clip

class ShemoDataset(Dataset):
    def __init__(self, df, processor, augment=False):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.augment   = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        file_path = row["path"]
        label     = LABEL2ID[row["label"]]

        # Load audio
        waveform, sr = torchaudio.load(file_path)

        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample to 16kHz
        if sr != TARGET_SR:
            waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)

        audio = waveform.squeeze().numpy()

        # Truncate or pad to 6 seconds
        if len(audio) > MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

        # Phone degradation on training data only
        if self.augment:
            audio = simulate_phone_audio(audio)

        inputs = self.processor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels":       torch.tensor(label, dtype=torch.long),
        }

print("Dataset class ready")

Dataset class ready


In [16]:
MODEL_CKPT = "facebook/wav2vec2-base"

processor = Wav2Vec2Processor.from_pretrained(MODEL_CKPT)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=NUM_LABELS,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

# Freeze feature extractor (renamed in transformers 4.40+)
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable params: {trainable:,}")
print(f"Frozen params:    {frozen:,}")
print("Model ready")

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 90,368,899
Frozen params:    4,200,448
Model ready


In [18]:
# Aggressive cleaning
df = df.dropna(subset=["path", "label"])
df = df[df["label"].isin(LABEL2ID.keys())]
df = df[df["path"].notna()]
df = df[df["path"] != ""]
df["label"] = df["label"].astype(str).str.strip()
df = df[df["label"].isin(LABEL2ID.keys())].reset_index(drop=True)

# Verify no NaN remains
print(f"NaN in label: {df['label'].isna().sum()}")
print(f"NaN in path:  {df['path'].isna().sum()}")
print(f"Clean samples: {len(df)}")
print(df["label"].value_counts())

NaN in label: 0
NaN in path:  0
Clean samples: 2513
label
calm          1229
frustrated    1059
disengaged     225
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42,
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")
print("\nTrain distribution:")
print(train_df["label"].value_counts())

train_dataset = ShemoDataset(train_df, processor, augment=True)
val_dataset   = ShemoDataset(val_df,   processor, augment=False)

# Class weights to handle disengaged (225 samples) being underrepresented
classes = np.array(list(LABEL2ID.keys()))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).cuda()
print(f"\nClass weights: {dict(zip(LABEL2ID.keys(), class_weights.round(3)))}")
print("Datasets ready")

Train: 2136  |  Val: 377

Train distribution:
label
calm          1045
frustrated     900
disengaged     191
Name: count, dtype: int64

Class weights: {'calm': np.float64(0.681), 'frustrated': np.float64(0.791), 'disengaged': np.float64(3.728)}
Datasets ready


In [20]:
@dataclass
class DataCollatorForAudio:
    processor: Wav2Vec2Processor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_values = [f["input_values"] for f in features]
        labels       = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch = self.processor.pad(
            {"input_values": input_values},
            padding=True,
            return_tensors="pt",
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorForAudio(processor=processor)

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average="weighted")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("Collator, metric, WeightedTrainer ready")

Collator, metric, WeightedTrainer ready


In [21]:
OUTPUT_DIR = "/kaggle/working/wav2vec2-shemo-bfsi"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=10,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    dataloader_num_workers=2,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,
    report_to="none",

    fp16=True,
    gradient_accumulation_steps=2,
)

print("Training arguments ready")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments ready


In [22]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,2.173972,1.057903,0.289928
2,1.376415,1.058539,0.271086
3,1.121492,1.050037,0.295968
4,0.997151,1.072490,0.306327
5,0.885158,1.087536,0.309537
6,0.650771,1.108665,0.302490
7,0.655082,1.149374,0.299551
8,0.539219,1.198244,0.292243
9,0.437857,1.244534,0.295968
10,0.412112,1.255068,0.299551


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=340, training_loss=0.9120192078983083, metrics={'train_runtime': 1579.7903, 'train_samples_per_second': 13.521, 'train_steps_per_second': 0.215, 'total_flos': 1.16352072110592e+18, 'train_loss': 0.9120192078983083, 'epoch': 10.0})

In [43]:
OUTPUT_DIR = "/kaggle/working/wav2vec2-shemo-bfsi"
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved


In [74]:
!zip -r my_model.zip /kaggle/working/wav2vec2-shemo-bfsi

  adding: kaggle/working/wav2vec2-shemo-bfsi/ (stored 0%)
  adding: kaggle/working/wav2vec2-shemo-bfsi/config.json (deflated 66%)
  adding: kaggle/working/wav2vec2-shemo-bfsi/processor_config.json (deflated 43%)
  adding: kaggle/working/wav2vec2-shemo-bfsi/tokenizer_config.json (deflated 73%)
  adding: kaggle/working/wav2vec2-shemo-bfsi/model.safetensors (deflated 9%)
  adding: kaggle/working/wav2vec2-shemo-bfsi/training_args.bin (deflated 53%)
  adding: kaggle/working/wav2vec2-shemo-bfsi/vocab.json (deflated 55%)


This Fine Tuning can be done better using IIITH Hindi Audio dataset combining with the Shemo one... For now we'll Go with the dataset we have now(SHEMO)